# Regressione logistica

In [1]:
import pandas as pd
import numpy as np
import re
import math
import time
from pathlib import Path
import warnings
from datetime import timedelta
from tabulate import tabulate
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer, classification_report
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare treining
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training

In [2]:
def training(file_path, csv_name):

    # Lettura dati
    df = pd.read_csv(file_path)

    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']
    
    # Rimuovo righe senza target validi
    df_validi = df.dropna(subset=original_target_list).copy()

    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    df_validi['HER2_class'] = (df_validi['HER2 [SII]'] >= 3).astype(int)

    final_target_list = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    features = features.fillna(features.mean())
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]

    cv = GroupKFold(n_splits=5)

    # Pipeline con Scaler: essenziale per la convergenza di 'saga' e l'efficacia di 'C'
    logistic_pipeline = Pipeline([
        ('scaler', RobustScaler()),
        ('classifier', LogisticRegression(
            random_state=42,
            n_jobs=1,
            class_weight='balanced', 
            solver='saga',          
            max_iter=3000           
        ))
    ])
    
    multi_output_model = MultiOutputClassifier(logistic_pipeline)

    # --- IPERPARAMETRI OTTIMIZZATI ---
    # Lista di dizionari: permette di testare l1_ratio solo quando penalty='elasticnet'
    iperparametri = [
        {
            'estimator__classifier__C': [0.01, 0.1, 1, 10],
            'estimator__classifier__penalty': ['l2']
        },
        {
            'estimator__classifier__C': [0.01, 0.1, 1, 10],
            'estimator__classifier__penalty': ['elasticnet'],
            'estimator__classifier__l1_ratio': [0.1, 0.5, 0.9]
        }
    ]

    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    # Calcolo combinazioni sommando le varianti per ogni dizionario nella lista
    total_combinations = sum(math.prod(len(v) for v in d.values()) for d in iperparametri)
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")

    # Ignoro warning di overflow specifici che possono capitare esplorando iperparametri estremi
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', 'overflow encountered')
        warnings.filterwarnings('ignore', 'invalid value encountered')

        grid_search = GridSearchCV(
            estimator=multi_output_model,
            param_grid=iperparametri,
            cv=cv,
            scoring=scorer,
            n_jobs=-1,
            verbose=1,
            refit=True,
            error_score=0.0 
        )

        grid_search.fit(features, target, groups=groups)

    # --- RECUPERO RISULTATI ---
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    fold_reports = []

    # Pulisco i nomi dei parametri per riutilizzarli nella pipeline pulita
    # Rimuovo 'estimator__classifier__' per ottenere es: 'C': 1, 'penalty': 'l2'
    clean_param_dict = {k.replace('estimator__classifier__', ''): v for k, v in best_params.items()}

    # Ricostruzione Pipeline per report dettagliato sui fold
    for train_idx, test_idx in cv.split(features, target, groups):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        # Ricreo la pipeline base e applico i parametri ottimali trovati
        fresh_pipeline = Pipeline([
            ('scaler', RobustScaler()),
            ('classifier', LogisticRegression(
                random_state=42, n_jobs=1, class_weight='balanced', solver='saga', max_iter=3000
            ))
        ])
        # Setto i parametri specifici del classifier (es: classifier__C)
        fresh_pipeline.set_params(**{f'classifier__{k}': v for k, v in clean_param_dict.items()})
        
        model_clone = MultiOutputClassifier(fresh_pipeline)
        model_clone.fit(X_train, y_train)
        y_pred = model_clone.predict(X_test)

        report_dict = {}
        for i, col in enumerate(final_target_list):
            report_dict[col] = classification_report(
                y_test.iloc[:, i],
                y_pred[:, i],
                output_dict=True,
                zero_division=0
            )
        fold_reports.append(report_dict)

    final_result = [{
        **clean_param_dict,
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports
    }]

    return final_result

# Stampo i risultati in un formato leggibile

In [3]:
def print_grid_search_results(results_per_dataset):

    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI (Logistic Regression)")
    print("=" * 80)

    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        best_result = metrics_list[0]

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")

        # Tabella iperparametri
        print("Iperparametri Ottimali:")
        possible_params = [
            ('C (Inverse Reg)', 'C'),
            ('Penalty', 'penalty'),
            ('L1 Ratio', 'l1_ratio')
        ]

        params_table = []
        for label, key in possible_params:
            # Uso .get() perché l1_ratio potrebbe non esistere se penalty è l2
            val = best_result.get(key, '-') 
            params_table.append([label, val])

        print(tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))
        print("\n Metriche di Classificazione per Target (Dettaglio primo fold):\n")
        
        target_names = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

        if best_result['fold_reports']:
            first_fold_report = best_result['fold_reports'][0]

            for target_name in target_names:
                if target_name not in first_fold_report: continue

                current_target_report = first_fold_report[target_name]
                rows = []
                classes = [c for c in ['0', '1'] if c in current_target_report]

                for cls in classes:
                    rows.append([
                        f"Classe {cls}",
                        f"{current_target_report[cls]['precision']:.3f}",
                        f"{current_target_report[cls]['recall']:.3f}",
                        f"{current_target_report[cls]['f1-score']:.3f}",
                        int(current_target_report[cls]['support'])
                    ])

                print(f"  Target: {target_name}")
                print(tabulate(rows, headers=['', 'Precision', 'Recall', 'F1-score', 'Support'],
                             tablefmt='simple', colalign=('left', 'center', 'center', 'center', 'center')))
                print()
        else:
            print("Nessun report dettagliato.")

        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            best_result.get('C'),
            best_result.get('penalty'),
            best_result.get('l1_ratio', '-')
        ])

    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    print(tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'C', 'Penalty', 'L1 Ratio'],
                   tablefmt='grid',
                   floatfmt=('.3f', '.3f', '.3f', '.0f', '.0f', '.0f')))

# Eseguo il tutto

In [4]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)



end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



Inizio Grid Search (16 combinazioni) per: t2_medsam
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-


Inizio Grid Search (16 combinazioni) per: t2_preprocessed
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-


Inizio Grid Search (16 combinazioni) per: t2_original
Fitting 5 folds for each of 16 candidates, totalling 80 fits


...


Inizio Grid Search (16 combinazioni) per: preprocessed_dynamic
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/fra


Inizio Grid Search (16 combinazioni) per: original_dynamic
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/fra


                         RIEPILOGO DEI MIGLIORI RISULTATI (Logistic Regression)

────────────────────────────────────────────────────────────────────────────────
 Dataset: t2_medsam
────────────────────────────────────────────────────────────────────────────────

 Performance: F1-score = 0.588 ± 0.057

Iperparametri Ottimali:
Parametro        Valore
---------------  ----------
C (Inverse Reg)  1
Penalty          elasticnet
L1 Ratio         0.5

 Metriche di Classificazione per Target (Dettaglio primo fold):

  Target: PR_class
           Precision    Recall    F1-score    Support
--------  -----------  --------  ----------  ---------
Classe 0     0.286      0.667       0.4          3
Classe 1      0.8       0.444      0.571         9

  Target: ER_class
           Precision    Recall    F1-score    Support
--------  -----------  --------  ----------  ---------
Classe 0      0.2         1        0.333         1
Classe 1       1        0.636      0.778        11

  Target: KI67_class
  